In [2]:
import os
import cv2
import json
import gzip
import pickle
import base64
import numpy as np
import re

from io import BytesIO
from PIL import Image
from openai import OpenAI
from collections import defaultdict

from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks


# ============================================================
# OPENAI
# ============================================================

os.environ["OPENAI_API_KEY"] = "YOUR-OPENAI-API-KEY"
client = OpenAI()


# ============================================================
# LOAD DATA
# ============================================================

video_file = "gym.mp4"

with gzip.open(
    "all_frames_data.pkl.gz",
    "rb"
) as f:

    all_frames_data = pickle.load(f)


# ============================================================
# VIDEO FPS
# ============================================================

cap = cv2.VideoCapture(video_file)

fps = cap.get(
    cv2.CAP_PROP_FPS
)

cap.release()

if fps is None or fps == 0:
    fps = 30

print("FPS:", fps)


# ============================================================
# TEMPORAL WINDOW
# Converts temporal window from seconds → number of frames
# ============================================================

TIME_WINDOW = 0.5

k = max(
    2,
    int(TIME_WINDOW * fps)
)

print("Temporal window k:", k)


# ============================================================
# HYPER-PARAMETERS
# ============================================================

# Max Image Size and Quality - sent to GPT
MAX_IMAGE_SIZE = 512
JPEG_QUALITY = 65

# Min distance between two peaks (peaks are major frame transitions (motion changes / scene changes))
PEAK_DISTANCE = int(1.0 * fps)

# Variance of Gaussian Filter to smoothen semantic transition curve
SMOOTH_SIGMA = 2

# Only retain TOP_K_PEAKS as input to GPT-4o
TOP_K_PEAKS = 12

# how far to move segment boundary during refinement
SEARCH_RADIUS = int(1.0 * fps)

# how far to look before/after a frame to compute semantic score
SEMANTIC_CONTEXT = int(0.4 * fps)

# Variance of Gaussian Filter to smoothen segment boundary during refinement
TRANSITION_SMOOTH_SIGMA = 2

# how many neighboring frames to include around each detected semantic peak
# frames within 0.25 sec are considered neighbors, min 3 frames included
PEAK_CONTEXT = max(
    3,
    int(0.25 * fps)
)

# used to compute number of anchor frames uniformly spread across video
# to capture frames other than peaks
GLOBAL_ANCHOR_DIVISOR = 80


# ============================================================
# SAFE JSON
# ============================================================

def safe_json(text):

    if text is None:
        raise ValueError("GPT returned None")

    text = text.strip()

    if len(text) == 0:
        raise ValueError("GPT returned empty response")

    try:
        return json.loads(text)

    except Exception:
        pass

    match = re.search(
        r"\{[\s\S]*\}",
        text
    )

    if match:

        candidate = match.group(0)

        try:
            return json.loads(candidate)

        except Exception:
            pass

    print("\n================ GPT RAW OUTPUT ================\n")
    print(text)
    print("\n================================================\n")

    raise ValueError("Invalid JSON")


# ============================================================
# FRAME READER
# ============================================================

def read_frame(video_file, frame_idx):

    cap = cv2.VideoCapture(video_file)

    cap.set(
        cv2.CAP_PROP_POS_FRAMES,
        int(frame_idx)
    )

    ret, frame = cap.read()

    cap.release()

    if not ret:
        return None

    frame = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )

    return Image.fromarray(frame)


# ============================================================
# IMAGE RESIZE
# ============================================================

def resize_for_gpt(
    img,
    max_size=512
):

    w, h = img.size

    scale = min(
        max_size / w,
        max_size / h
    )

    new_w = int(w * scale)
    new_h = int(h * scale)

    return img.resize(
        (new_w, new_h),
        Image.LANCZOS
    )


# ============================================================
# PIL → DATA URL
# Converts PIL image to base64-encoded data url
# ============================================================

def pil_to_data_url(img):

    buffer = BytesIO()

    img.save(
        buffer,
        format="JPEG",
        quality=JPEG_QUALITY
    )

    b64 = base64.b64encode(
        buffer.getvalue()
    ).decode()

    return (
        f"data:image/jpeg;base64,{b64}"
    )


# ============================================================
# LOAD FRAMES
# Loads specified frame_ids and prepares them for GPT processing
# ============================================================

def load_selected_frames(frame_ids):

    images = []
    valid_ids = []

    for fid in frame_ids:

        img = read_frame(
            video_file,
            fid
        )

        if img is None:
            continue

        img = resize_for_gpt(
            img,
            MAX_IMAGE_SIZE
        )

        images.append(img)
        valid_ids.append(int(fid))

    return images, valid_ids


# ============================================================
# FEATURE EXTRACTION
# Track Wise Feature Extraction
# Features = Object Embeddings, Motion Mag, Motion Dir
# ============================================================

tracks = defaultdict(list)

for frame_data in all_frames_data:

    for obj in frame_data:

        if obj.get("embedding") is None:
            continue

        track_id = obj["track_id"]

        emb = obj["embedding"]

        x1, y1, x2, y2 = obj["bbox"]

        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2

        tracks[track_id].append({

            "embedding": emb,

            "center": np.array([
                cx,
                cy
            ])

        })


track_features = {}

# seq = temporal sequence of observations for an object / track_id
for track_id, seq in tracks.items():

    # ignore short tracks - insufficient data
    if len(seq) < k + 5:
        continue

    embs = np.stack([
        s["embedding"]
        for s in seq
    ])

    centers = np.stack([
        s["center"]
        for s in seq
    ])

    # ========================================================
    # normalize embeddings
    # L2 norm, 1e-6 to avoid near zero norms
    # ========================================================

    embs = embs / (
        np.linalg.norm(
            embs,
            axis=1,
            keepdims=True
        ) + 1e-6
    )

    # ========================================================
    # motion
    # ========================================================

    motion = np.diff(
        centers,
        axis=0
    )

    # (1, 0) = Add 1 row before 0th element, Add 0 rows after last element
    # (0, 0) = Keep columns unchanged
    # Padded element is same as 0th element, "edge" = use edge element for padding
    motion = np.pad(
        motion,
        ((1, 0), (0, 0)),
        mode="edge"
    )

    motion_mag = np.linalg.norm(
        motion,
        axis=1
    )

    motion_dir = motion / (
        np.linalg.norm(
            motion,
            axis=1,
            keepdims=True
        ) + 1e-6
    )

    track_features[track_id] = {

        "embeddings": embs,

        "motion_mag": motion_mag,

        "motion_dir": motion_dir

    }


print(
    "Tracks:",
    list(track_features.keys())
)


# ============================================================
# SEMANTIC KEYFRAME SAMPLING
# Samples 1st frame, last frame and peaks
# ============================================================

def semantic_keyframe_sampling(
    embs,
    motion_mag
):

    # embedding difference = scene change
    emb_diff = np.linalg.norm(
        embs[1:] - embs[:-1],
        axis=1
    )

    emb_diff = np.pad(
        emb_diff,
        (1, 0),
        mode="edge"
    )

    semantic_score = (
        0.8 * emb_diff +
        0.2 * motion_mag
    )

    semantic_score = gaussian_filter1d(
        semantic_score,
        sigma=SMOOTH_SIGMA
    )

    peaks, _ = find_peaks(
        semantic_score,
        distance=PEAK_DISTANCE
    )

    if len(peaks) > TOP_K_PEAKS:

        strongest = np.argsort(
            semantic_score[peaks]
        )[-TOP_K_PEAKS:]

        peaks = peaks[strongest]

    peaks = sorted([
        int(p)
        for p in peaks.tolist()
    ])

    # Add 1st frame
    final_ids = [0]

    # Add Peak Frames
    final_ids.extend(peaks)

    # Add last frame
    final_ids.append(
        len(embs) - 1
    )

    # sort chronologically, duplicates removed
    final_ids = sorted([
        int(x)
        for x in set(final_ids)
    ])

    return final_ids


# ============================================================
# EXPAND TEMPORAL CONTEXT
# expanded = final set of frames fed to GPT-4o
# includes 1st frame, last frame, peak frames,
# neighbors of peaks, anchor frames
# ============================================================

def expand_keyframes(
    peak_ids,
    total_frames
):

    expanded = set()

    # ========================================================
    # adaptive global anchors
    # ========================================================

    n_anchors = max(
        4,
        total_frames // GLOBAL_ANCHOR_DIVISOR
    )

    anchors = np.linspace(
        0,
        total_frames - 1,
        n_anchors
    ).astype(int)

    for a in anchors:
        expanded.add(int(a))

    # ========================================================
    # temporal context around peaks
    # ========================================================

    for p in peak_ids:

        expanded.add(int(p))

        expanded.add(
            max(
                0,
                int(p - PEAK_CONTEXT)
            )
        )

        expanded.add(
            min(
                total_frames - 1,
                int(p + PEAK_CONTEXT)
            )
        )

    expanded = sorted([
        int(x)
        for x in expanded
    ])

    return expanded


# ============================================================
# GPT SEGMENTATION
# ============================================================

def gpt_coarse_segmentation(
    images,
    frame_ids,
    total_frames
):

    content = []

    prompt = f"""
You are analyzing temporally ordered keyframes from a video.

Each image corresponds to a frame in time order.

Your task:
Segment the video into coherent temporal activity segments.

A segment represents a period where:
- the main action is consistent
- the scene context is stable
- the intent or behavior does not change significantly

IMPORTANT PRINCIPLES:

1. Temporal coherence
- Do not split segments for minor visual changes
- Only create a new segment when there is a meaningful shift in activity, context, or intent

2. Visual grounding
- Base decisions ONLY on visible information
- Avoid guessing hidden intent or unseen actions

3. Granularity control
- Prefer fewer, more meaningful segments over many small ones
- Avoid micro-segmentation of continuous actions

4. Transition handling
- Early frames may represent setup or preparation
- Later frames may represent completion or transition out of activity

Return STRICT JSON ONLY:

{{
  "segments": [
    {{
      "start_frame": 0,
      "end_frame": 120,
      "activity": "short human-readable description of activity"
    }}
  ]
}}

RULES:
- Segments must fully cover the video [0 → {total_frames-1}]
- No overlaps between segments
- No gaps between segments
- Segments must be ordered in time
- Final segment must end at {total_frames - 1}
"""

    content.append({
        "type": "input_text",
        "text": prompt
    })

    for img, fid in zip(
        images,
        frame_ids
    ):

        content.append({
            "type": "input_text",
            "text": f"Frame {fid}"
        })

        content.append({
            "type": "input_image",
            "image_url":
                pil_to_data_url(img)
        })

    response = client.responses.create(
        model="gpt-4o",
        input=[{
            "role": "user",
            "content": content
        }]
    )

    return safe_json(
        response.output_text
    )


# ============================================================
# COSINE SIMILARITY
# ============================================================

def cosine_similarity(a, b):

    return np.dot(a, b) / (
        np.linalg.norm(a) *
        np.linalg.norm(b) + 1e-6
    )


# ============================================================
# SEMANTIC TRANSITION SCORE
# ============================================================

def semantic_transition_score(
    embs,
    motion_dir,
    motion_mag
):

    n = len(embs)

    score = np.zeros(n)

    for t in range(
        SEMANTIC_CONTEXT,
        n - SEMANTIC_CONTEXT
    ):

        emb_before = embs[
            t - SEMANTIC_CONTEXT
        ]

        emb_after = embs[
            t + SEMANTIC_CONTEXT
        ]

        emb_change = np.linalg.norm(
            emb_after - emb_before
        )

        dir_before = motion_dir[t - 1]
        dir_after = motion_dir[t]

        dir_change = (
            1 -
            cosine_similarity(
                dir_before,
                dir_after
            )
        )

        mag_change = abs(
            motion_mag[t] -
            motion_mag[t - 1]
        )

        score[t] = (
            0.6 * emb_change +
            0.3 * dir_change +
            0.1 * mag_change
        )

    score = gaussian_filter1d(
        score,
        sigma=TRANSITION_SMOOTH_SIGMA
    )

    return score


# ============================================================
# LOCAL SEMANTIC REFINEMENT
# ============================================================

def refine_boundaries(
    embs,
    motion_dir,
    motion_mag,
    gpt_segments
):

    transition_score = semantic_transition_score(
        embs,
        motion_dir,
        motion_mag
    )

    refined_boundaries = []

    rough_boundaries = []

    for seg in gpt_segments[:-1]:

        rough_boundaries.append(
            int(seg["end_frame"])
        )

    n = len(embs)

    for i, boundary in enumerate(
        rough_boundaries
    ):

        # ====================================================
        # midpoint constraints
        # prevents refined boundary overlaps
        # refined boundaries don't cross mid-points of coarse boundaries
        # ====================================================

        if i == 0:

            left_limit = 0

        else:

            prev_b = rough_boundaries[i - 1]

            left_limit = (
                prev_b + boundary
            ) // 2

        if i == len(rough_boundaries) - 1:

            right_limit = n - 1

        else:

            next_b = rough_boundaries[i + 1]

            right_limit = (
                boundary + next_b
            ) // 2

        # ====================================================
        # clipped search window
        # only search near GPT coarse boundary
        # GPT semantics should be respected, search should remain local
        # ====================================================

        left = max(
            left_limit,
            boundary - SEARCH_RADIUS
        )

        right = min(
            right_limit,
            boundary + SEARCH_RADIUS
        )

        # ====================================================
        # local transition scores
        # semantic_transition_scores in local window around coarse boundary
        # ====================================================

        local_scores = transition_score[
            left:right+1
        ]

        # ====================================================
        # detect local peaks
        # ====================================================

        peaks, _ = find_peaks(
            local_scores
        )

        # ====================================================
        # fallback if no peaks
        # ====================================================

        # if no peaks, choose strongest semantic score as peak in the local window
        if len(peaks) == 0:

            best_local = np.argmax(
                local_scores
            )

            refined = left + best_local

        else:

            candidate_boundaries = []

            for p in peaks:

                global_pos = left + int(p)

                # --------------------------------------------
                # distance from GPT boundary
                # --------------------------------------------

                dist = abs(
                    global_pos - boundary
                )

                # --------------------------------------------
                # transition strength
                # --------------------------------------------

                strength = local_scores[p]

                # --------------------------------------------
                # score = how close are we to GPT boundary and how
                # strong (peaky) is the peak
                # --------------------------------------------

                score = (
                    strength
                    -
                    0.02 * dist
                )

                candidate_boundaries.append({

                    "score": score,

                    "position": global_pos,

                    "distance": dist,

                    "strength": strength

                })

            # =================================================
            # choose best candidate
            # =================================================

            candidate_boundaries = sorted(
                candidate_boundaries,
                key=lambda x: x["score"],
                reverse=True
            )

            refined = candidate_boundaries[0][
                "position"
            ]

        # ====================================================
        # enforce non-overlapping refined boundaries
        # current segment start is at least one frame after
        # prev segment end
        # ====================================================

        if len(refined_boundaries) > 0:

            refined = max(
                refined,
                refined_boundaries[-1] + 1
            )

        refined_boundaries.append(
            int(refined)
        )

    return refined_boundaries


# ============================================================
# BUILD FINAL SEGMENTS
# Convert refined transition points into segments
# ============================================================

def build_final_segments(
    refined_boundaries,
    total_frames
):

    segments = []

    current_start = 0

    for boundary in refined_boundaries:

        segments.append((
            int(current_start),
            int(boundary)
        ))

        current_start = boundary + 1

    segments.append((
        int(current_start),
        int(total_frames - 1)
    ))

    return segments


# ============================================================
# SAMPLE SEGMENT FRAMES
# Uniformly sample each segment to be sent to GPT
# ============================================================

def sample_segment_frames(
    segment,
    n_samples=7
):

    start, end = segment

    ids = [
        int(x)
        for x in np.linspace(
            start,
            end,
            n_samples
        )
    ]

    images = []
    valid_ids = []

    for fid in ids:

        img = read_frame(
            video_file,
            fid
        )

        if img is None:
            continue

        img = resize_for_gpt(
            img,
            MAX_IMAGE_SIZE
        )

        images.append(img)
        valid_ids.append(int(fid))

    return images, valid_ids


# ============================================================
# STORY UNDERSTANDING
# ============================================================

def analyze_video_story(
    segments
):

    content = []

    prompt = """
You are analyzing a temporally segmented video.

Each segment contains frames that belong to a single coherent activity.

Your task:
For each segment, describe:
1. What the main activity is
2. What is visually happening
3. How the activity evolves if relevant

IMPORTANT PRINCIPLES:

1. Temporal consistency
- Each segment should represent one stable activity or phase
- Do not split or merge segments mentally

2. Visual grounding
- Only describe what is clearly visible
- Avoid guessing unseen actions or intent

3. Activity transitions
- Earlier segments may represent setup / introduction
- Middle segments represent the main activity
- Later segments may represent transition, completion, or rest

4. Granularity control
- Use natural human-level activity descriptions
- Do not over-fragment into micro-actions
- Do not over-generalize into vague labels

5. Consistency rule
- If an activity continues across segments, keep naming consistent
- If it changes meaningfully, reflect that change clearly

Return STRICT JSON ONLY:

{
  "overall_video": "One-sentence summary of the entire video",
  "segments": [
    {
      "segment_id": 0,
      "activity": "short descriptive label of activity",
      "about": "clear explanation grounded in visual evidence"
    }
  ]
}
"""

    content.append({
        "type": "input_text",
        "text": prompt
    })

    for seg in segments:

        content.append({
            "type": "input_text",
            "text":
            f"""
Segment {seg['segment_id']}
Frames: {seg['frames']}
"""
        })

        for img in seg["images"]:

            content.append({
                "type": "input_image",
                "image_url":
                    pil_to_data_url(img)
            })

    response = client.responses.create(
        model="gpt-4o",
        input=[{
            "role": "user",
            "content": content
        }]
    )

    return safe_json(
        response.output_text
    )


# ============================================================
# MAIN PIPELINE
# ============================================================

final_results = {}

for track_id, feats in track_features.items():

    embs = feats["embeddings"]
    motion_mag = feats["motion_mag"]
    motion_dir = feats["motion_dir"]

    if len(embs) < 30:
        continue

    print("\n===================================")
    print("TRACK:", track_id)
    print("===================================")

    # ========================================================
    # SEMANTIC PEAKS
    # ========================================================

    peak_ids = semantic_keyframe_sampling(
        embs,
        motion_mag
    )

    print("\nPeak Keyframes:")
    print(peak_ids)

    # ========================================================
    # EXPANDED CONTEXT
    # ========================================================

    expanded_ids = expand_keyframes(
        peak_ids,
        len(embs)
    )

    print("\nExpanded Keyframes:")
    print(expanded_ids)

    # ========================================================
    # LOAD FRAMES
    # ========================================================

    images, frame_ids = load_selected_frames(
        expanded_ids
    )

    # ========================================================
    # GPT SEGMENTATION
    # ========================================================

    coarse = gpt_coarse_segmentation(
        images,
        frame_ids,
        len(embs)
    )

    print("\nGPT Coarse Segments:")
    print(
        json.dumps(
            coarse,
            indent=2
        )
    )

    # ========================================================
    # REFINE BOUNDARIES
    # ========================================================

    refined_boundaries = refine_boundaries(
        embs,
        motion_dir,
        motion_mag,
        coarse["segments"]
    )

    print("\nRefined Boundaries:")
    print(refined_boundaries)

    # ========================================================
    # FINAL SEGMENTS
    # ========================================================

    final_segments = build_final_segments(
        refined_boundaries,
        len(embs)
    )

    print("\nFinal Segments:")
    print(final_segments)

    # ========================================================
    # PREP SEGMENT INPUTS
    # ========================================================

    segment_inputs = []

    for idx, segment in enumerate(
        final_segments
    ):

        seg_images, seg_frames = (
            sample_segment_frames(
                segment,
                n_samples=7
            )
        )

        segment_inputs.append({

            "segment_id": idx,
            "frames": seg_frames,
            "images": seg_images

        })

    # ========================================================
    # STORY UNDERSTANDING
    # ========================================================

    story = analyze_video_story(
        segment_inputs
    )

    print("\n===================================")
    print("FINAL STORY")
    print("===================================")

    print(
        json.dumps(
            story,
            indent=2
        )
    )

    # ========================================================
    # SAVE RESULTS
    # ========================================================

    final_results[track_id] = {

        "peak_keyframes":
            peak_ids,

        "expanded_keyframes":
            expanded_ids,

        "gpt_segments":
            coarse["segments"],

        "refined_boundaries":
            refined_boundaries,

        "final_segments":
            final_segments,

        "story":
            story
    }

FPS: 30.0
Temporal window k: 15
Tracks: [1]

TRACK: 1

Peak Keyframes:
[0, 33, 67, 130, 164, 225, 285, 338, 385, 443, 473, 479]

Expanded Keyframes:
[0, 7, 26, 33, 40, 60, 67, 74, 95, 123, 130, 137, 157, 164, 171, 191, 218, 225, 232, 278, 285, 287, 292, 331, 338, 345, 378, 383, 385, 392, 436, 443, 450, 466, 472, 473, 479]

GPT Coarse Segments:
{
  "segments": [
    {
      "start_frame": 0,
      "end_frame": 120,
      "activity": "Seated weightlifting exercises with consistent form"
    },
    {
      "start_frame": 121,
      "end_frame": 336,
      "activity": "Dumbbell lateral raises with dynamic motions"
    },
    {
      "start_frame": 337,
      "end_frame": 472,
      "activity": "Bent-over dumbbell raises for shoulder workout"
    },
    {
      "start_frame": 473,
      "end_frame": 479,
      "activity": "Cool down and transition with talking"
    }
  ]
}

Refined Boundaries:
[130, 336, 464]

Final Segments:
[(0, 130), (131, 336), (337, 464), (465, 479)]

FINAL STORY
{
  "